In [1]:
import timm
import cv2
import torch

c:\Users\aseva\Desktop\MyEDU\YaDLE\YaDLE_gensim_vscode_stream_4\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
total_models = len(timm.list_models())
print(total_models)
print(timm.list_models(filter="resnet*")[:5])

1288
['resnet10t', 'resnet14t', 'resnet18', 'resnet18d', 'resnet26']


In [3]:
model = timm.create_model(model_name="resnet18")
print(model) 

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act2): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, m

**Основные параметры, которые могут быть полезны:**
* `tag` — как правило, содержит информацию об обучении модели. В частности, здесь a1_in1k — означает, что модель обучалась на датасете ImageNet с 1000 классами.
* `input_size` — размер изображений, использованный при тренировке.
* `fixed_input_size` — атрибут, указывающий на жёсткость ограничения входного размера изображений. В данном случае False означает, что ограничений нет.
* `mean` и `std` — коэффициенты для нормализации изображений, использованные при тренировке.

**NB!:** По конфигурационному файлу модели можно понять, какую предобработку она требует. Это удобно для ранее не известных вам моделей. 

In [4]:
model = timm.create_model(model_name="resnet18", pretrained=True)
model.pretrained_cfg 

{'url': 'https://github.com/huggingface/pytorch-image-models/releases/download/v0.1-rsb-weights/resnet18_a1_0-d63eafa0.pth',
 'hf_hub_id': 'timm/resnet18.a1_in1k',
 'architecture': 'resnet18',
 'tag': 'a1_in1k',
 'custom_load': False,
 'input_size': (3, 224, 224),
 'test_input_size': (3, 288, 288),
 'fixed_input_size': False,
 'interpolation': 'bicubic',
 'crop_pct': 0.95,
 'test_crop_pct': 1.0,
 'crop_mode': 'center',
 'mean': (0.485, 0.456, 0.406),
 'std': (0.229, 0.224, 0.225),
 'num_classes': 1000,
 'pool_size': (7, 7),
 'first_conv': 'conv1',
 'classifier': 'fc',
 'license': 'apache-2.0',
 'origin_url': 'https://github.com/huggingface/pytorch-image-models',
 'paper_ids': 'arXiv:2110.00476'}

# Задание 2
Оцените качество предобученной модели `maxvit_large_tf_384` из библиотеки `timm`. На примере одного, уже сохранённого ранее изображения проинициализируйте модель и подготовьте картинку к подаче на вход модели: 
* Загрузите.
* Масштабируйте в нужный формат.
* Нормализуйте сначала в диапазон от 0 до 1, а затем в формат, использованный при тренировке конкретной модели.
* Выведите результирующую размерность изображения.
* Используйте атрибуты `input_size | mean | std` из конфига модели `pretrained_cfg`.

In [5]:
#model = timm.create_model("maxvit_large_tf_384", pretrained=True)

In [6]:
img = cv2.imread("../data/eskimo_dog.jpg")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [7]:
# выбираем H x W для масштабирования и конвертируем в массив
img = cv2.resize(img, model.pretrained_cfg["input_size"][1:])

# нормализуем в диапазон [0,1], а затем применяем нормализацию для модели
img = img / 255  
img = (img - model.pretrained_cfg["mean"]) / model.pretrained_cfg["std"]

# H x W x C -> C x H x W
img = img.transpose(2, 0, 1)
print(img.shape)

(3, 224, 224)


Ещё одна важная часть возможностей `TIMM` связана с инициализацией самой модели.
Большая часть моделей обучались на задачу классификации и построены по упрощённому принципу:
* последовательные слои конволюций,
* агрегирование значений карт признаков,
* линейный классификатор.

Для конечной задачи не всегда интересен результирующий классификатор — больше наиболее важны признаки модели (тензора до линейного слоя). Производить `feature-extraction` в `timm` легко — достаточно указать `features_only=True`. 
Посмотрим на примере модели `efficientnet_b0`. Извлечём только признаки и посмотрим на их размерность. Для этого пропустим через модель тензор из случайных значений:

In [8]:
model = timm.create_model("efficientnet_b0",
                          pretrained=True,
                          features_only=True)

x = torch.randn(1, 3, 224, 224)
out = model(x)
print()
[print(x.shape) for x in out] 

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.



torch.Size([1, 16, 112, 112])
torch.Size([1, 24, 56, 56])
torch.Size([1, 40, 28, 28])
torch.Size([1, 112, 14, 14])
torch.Size([1, 320, 7, 7])


[None, None, None, None, None]

Количеством извлекаемых признаков можно управлять с помощью параметра out_indices, в который передаётся список индексов, соответствующий блокам признаков. Для некоторых задач могут быть релевантны только признаки с самых глубоких слоёв — и для модели из примера выше мы могли бы извлечь, например, только 2 последних признака. Чтобы понять общее количество возвращаемых признаков для конкретной модели, можно сначала указать только features_only, а по количеству тензоров понять, какие индексы можно указывать.

In [9]:
model = timm.create_model("efficientnet_b0",
                          pretrained=True,
                          features_only=True,
                          out_indices=[3, 4])

x = torch.randn(1, 3, 224, 224)
out = model(x)
print()
[print(x.shape) for x in out] 

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.



torch.Size([1, 112, 14, 14])
torch.Size([1, 320, 7, 7])


[None, None]